In [1]:
import polars as pl
import os
import kagglehub

/home/vinicius/Documents/Workspace/ml_data_engeneering/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [73]:
def check_primary_key(df:pl.DataFrame, cols:list[str]|str)->bool:
    return len(df) == len(df.unique(cols))

In [2]:
dataset_dir = "./dataset"
if not os.path.isdir(dataset_dir):
    os.mkdir(dataset_dir)

In [4]:
path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

100%|██████████| 42.6M/42.6M [00:02<00:00, 18.2MB/s]

Extracting files...


Path to dataset files: /home/vinicius/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


In [5]:
os.listdir(path)

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [7]:
csv_files = [os.path.join(path, f) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f)) and f.endswith(".csv")]
len(csv_files)

9

In [11]:
from shutil import copy2

for file in csv_files:
    dst_path = os.path.join(
        dataset_dir,
        os.path.split(file)[-1]
    )
    copy2(src=file, dst=dst_path)

In [ ]:
customer_df = pl.read_csv(os.path.join(dataset_dir, 'olist_customers_dataset.csv'))
orders_df = pl.read_csv(os.path.join(dataset_dir,'olist_orders_dataset.csv'))
geolocation_df = pl.read_csv(os.path.join(dataset_dir,'olist_geolocation_dataset.csv'))
items_df = pl.read_csv(os.path.join(dataset_dir,'olist_order_items_dataset.csv'))
payments_df = pl.read_csv(os.path.join(dataset_dir,'olist_order_payments_dataset.csv'))
reviews_df = pl.read_csv(os.path.join(dataset_dir,'olist_order_reviews_dataset.csv'))
products_df = pl.read_csv(os.path.join(dataset_dir,'olist_products_dataset.csv'))
sellers_df = pl.read_csv(os.path.join(dataset_dir,'olist_sellers_dataset.csv'))
category_translation_df = pl.read_csv(os.path.join(dataset_dir,'product_category_name_translation.csv'))

# CUSTOMERS_DATASET:

- customer_df:
    - <u>customer_id</u>: char(32), 
    - customer_unique_id: char(32), 
    - customer_zip_code_prefix: float
    - customer_city: varchar(50),
    - customer_state: char(2)
        

In [17]:
customer_df.describe()


statistic,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
str,str,str,f64,str,str
"""count""","""99441""","""99441""",99441.0,"""99441""","""99441"""
"""null_count""","""0""","""0""",0.0,"""0""","""0"""
"""mean""",null,null,35137.474583,null,null
"""std""",null,null,29797.938996,null,null
"""min""","""00012a2ce6f8dcda20d059ce984917…","""0000366f3b9a7992bf8c76cfdf3221…",1003.0,"""abadia dos dourados""","""AC"""
"""25%""",null,null,11347.0,null,null
"""50%""",null,null,24416.0,null,null
"""75%""",null,null,58900.0,null,null
"""max""","""ffffe8b65bbe3087b653a978c870db…","""ffffd2657e2aad2907e67c3e9daecb…",99990.0,"""zortea""","""TO"""


In [18]:
customer_df.head()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
str,str,i64,str,str
"""06b8999e2fba1a1fbc88172c00ba8b…","""861eff4711a542e4b93843c6dd7feb…",14409,"""franca""","""SP"""
"""18955e83d337fd6b2def6b18a428ac…","""290c77bc529b7ac935b93aa66c333d…",9790,"""sao bernardo do campo""","""SP"""
"""4e7b3e00288586ebd08712fdd0374a…","""060e732b5b29e8181a18229c7b0b2b…",1151,"""sao paulo""","""SP"""
"""b2b6027bc5c5109e529d4dc6358b12…","""259dac757896d24d7702b9acbbff3f…",8775,"""mogi das cruzes""","""SP"""
"""4f2d8ab171c80ec8364f7c12e35b23…","""345ecd01c38d18a9036ed96c73b8d0…",13056,"""campinas""","""SP"""


In [27]:
for col in customer_df.schema:
    if customer_df.schema[col] == pl.String:
        print(max(customer_df.with_columns(len_col=pl.col(col).str.len_chars())["len_col"]))

32
32
32
2


In [75]:
# check primary key:
check_primary_key(customer_df, "customer_id")


True

# ORDER_DATASET

- orders_df:
    - <u>order_id</u>: char(32)
    - customer_id: char(32) -> FK.customer_df.customer_id
    - order_status: varchar(20)
    - order_purchase_timestamp: timestamp
    - order_aproved_at: timestamp
    - order_delivered_carrier_date: timestamp
    - order_delivered_customer_date: timestamp
    - order_estimated_delivery_date: date

In [44]:
orders_df.describe()

statistic,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str,str
"""count""","""99441""","""99441""","""99441""","""99441""","""99281""","""97658""","""96476""","""99441"""
"""null_count""","""0""","""0""","""0""","""0""","""160""","""1783""","""2965""","""0"""
"""mean""",null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null
"""min""","""00010242fe8c5a6d1ba2dd792cb162…","""00012a2ce6f8dcda20d059ce984917…","""approved""","""2016-09-04 21:15:19""","""2016-09-15 12:16:38""","""2016-10-08 10:34:01""","""2016-10-11 13:46:32""","""2016-09-30 00:00:00"""
"""25%""",null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null
"""max""","""fffe41c64501cc87c801fd61db3f62…","""ffffe8b65bbe3087b653a978c870db…","""unavailable""","""2018-10-17 17:30:18""","""2018-09-03 17:40:06""","""2018-09-11 19:48:28""","""2018-10-17 13:22:46""","""2018-11-12 00:00:00"""


In [48]:
orders_df.group_by("order_status").len().with_columns(prop=(pl.col("len")/len(orders_df)).round(4)).sort("order_status")

order_status,len,prop
str,u32,f64
"""approved""",2,0.0
"""canceled""",625,0.0063
"""created""",5,0.0001
"""delivered""",96478,0.9702
"""invoiced""",314,0.0032
"""processing""",301,0.003
"""shipped""",1107,0.0111
"""unavailable""",609,0.0061


In [76]:
# check primary key:
check_primary_key(orders_df, "order_id")

True

# GEOLOCATION_DATASET

- geolocation_df:
    - geolocation_zip_code_prefix: integer
    - geolocation_lat: decimal(8, 6)
    - geolocation_lng: decimal(9, 6)
    - geolocation_city: varchar(50)
    - geolocation_state: char(2)

In [54]:
geolocation_df.describe()

statistic,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
str,f64,f64,f64,str,str
"""count""",1.000163e6,1.000163e6,1.000163e6,"""1000163""","""1000163"""
"""null_count""",0.0,0.0,0.0,"""0""","""0"""
"""mean""",36574.166466,-21.176153,-46.390541,null,null
"""std""",30549.33571,5.715866,4.269748,null,null
"""min""",1001.0,-36.605374,-101.466766,"""* cidade""","""AC"""
"""25%""",11075.0,-23.603545,-48.573137,null,null
"""50%""",26530.0,-22.919377,-46.637879,null,null
"""75%""",63504.0,-19.979614,-43.767703,null,null
"""max""",99990.0,45.065933,121.105394,"""óleo""","""TO"""


In [59]:
# Check if geolocation_zip_code_prefix is float or int
if len(geolocation_df.filter(pl.col("geolocation_zip_code_prefix") % 1 != 0)) == 0:
    print("Integer")
else:
    print("Float")

Integer


In [77]:
# check if any columns is unique:
for col in geolocation_df.columns:
    print(f"{col} -> is_unique: {check_primary_key(geolocation_df, col)}")


geolocation_zip_code_prefix -> is_unique: False
geolocation_lat -> is_unique: False
geolocation_lng -> is_unique: False
geolocation_city -> is_unique: False
geolocation_state -> is_unique: False


In [78]:
# Check possible pk:
check_primary_key(geolocation_df, ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])

False

# ORDER ITEMS DATASET:

- items_df:
    - <u>order_id</u>: char(32) -> FK payment_df.order_id
    - <u>order_item_id</u>: int
    - product_id: char(32) -> FK product_df.product_id
    - seller_id : char(32) -> FK sellers_df.seller_id
    - shipping_limit_date: timestamp
    - price: decimal(10, 2)
    - freight_value: decimal(10, 2)

In [66]:
items_df.describe()

statistic,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,str,f64,str,str,str,f64,f64
"""count""","""112650""",112650.0,"""112650""","""112650""","""112650""",112650.0,112650.0
"""null_count""","""0""",0.0,"""0""","""0""","""0""",0.0,0.0
"""mean""",null,1.197834,null,null,null,120.653739,19.99032
"""std""",null,0.705124,null,null,null,183.633928,15.806405
"""min""","""00010242fe8c5a6d1ba2dd792cb162…",1.0,"""00066f42aeeb9f3007548bb9d3f33c…","""0015a82c2db000af6aaaf3ae2ecb05…","""2016-09-19 00:15:34""",0.85,0.0
"""25%""",null,1.0,null,null,null,39.9,13.08
"""50%""",null,1.0,null,null,null,74.99,16.26
"""75%""",null,1.0,null,null,null,134.9,21.15
"""max""","""fffe41c64501cc87c801fd61db3f62…",21.0,"""fffe9eeff12fcbd74a2f2b007dde0c…","""ffff564a4f9085cd26170f47323937…","""2020-04-09 22:35:08""",6735.0,409.68


In [79]:
check_primary_key(items_df, ["order_id", "order_item_id"])

True

# ORDER PAYMENTS DATASET:

- payments_df:
    - <u>order_id</u>: char(32)
    - <u>payment_sequential</u>: int
    - payment_type: varchar(50)
    - payment_installments: int
    - payment_value: decimal(10, 2)

In [70]:
payments_df.describe()

statistic,order_id,payment_sequential,payment_type,payment_installments,payment_value
str,str,f64,str,f64,f64
"""count""","""103886""",103886.0,"""103886""",103886.0,103886.0
"""null_count""","""0""",0.0,"""0""",0.0,0.0
"""mean""",null,1.092679,null,2.853349,154.10038
"""std""",null,0.706584,null,2.687051,217.494064
"""min""","""00010242fe8c5a6d1ba2dd792cb162…",1.0,"""boleto""",0.0,0.0
"""25%""",null,1.0,null,1.0,56.79
"""50%""",null,1.0,null,1.0,100.0
"""75%""",null,1.0,null,4.0,171.84
"""max""","""fffe41c64501cc87c801fd61db3f62…",29.0,"""voucher""",24.0,13664.08


In [84]:
check_primary_key(payments_df, ["order_id", "payment_sequential"])

True

# REVIEWS DATASET

- reviews_df:
    - <u>review_id</u>: char(32)
    - <u>order_id</u>: char(32) -> FK order_df.order_id
    - review_score: int
    - review_comment_title: varchar(255) CHARACTER SET utf8mb4
    - review_comment_message: varchar(2000) CHARACTER SET utf8mb4
    - review_creation_date: timestamp
    - review_answer_timestamp: timestamp

In [72]:
reviews_df.describe()

statistic,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,str,f64,str,str,str,str
"""count""","""99224""","""99224""",99224.0,"""11568""","""40977""","""99224""","""99224"""
"""null_count""","""0""","""0""",0.0,"""87656""","""58247""","""0""","""0"""
"""mean""",null,null,4.086421,null,null,null,null
"""std""",null,null,1.347579,null,null,null,null
"""min""","""0001239bc1de2e33cb583967c2ca4c…","""00010242fe8c5a6d1ba2dd792cb162…",1.0,""" """,""" ""","""2016-10-02 00:00:00""","""2016-10-07 18:32:28"""
"""25%""",null,null,4.0,null,null,null,null
"""50%""",null,null,5.0,null,null,null,null
"""75%""",null,null,5.0,null,null,null,null
"""max""","""fffefe7a48d22f7b32046421062219…","""fffe41c64501cc87c801fd61db3f62…",5.0,"""🔟 ""","""😡😡😡😡😡👎👎👎👎👎 Empresa sem compro…","""2018-08-31 00:00:00""","""2018-10-29 12:27:35"""


In [86]:
# Check primary_key
check_primary_key(reviews_df, ["review_id", "order_id"])

True

In [87]:
len(reviews_df.filter(pl.col("review_score")%1!=0))

0

# PRODUCT DATASET

- product_df:
    - <u>product_id</u>: char(32)
    - product_category_name: varchar(200)
    - product_name_lenght: int
    - product_description_lenght: int 
    - product_photos_qty: int
    - product_weight_g: decimal(8, 2)
    - product_lenght_cm: decimal(8,2)
    - product_heigh_cm: decimal(8,2)
    - product_width_cm: decimal(8,2)


In [88]:
products_df.describe()

statistic,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,str,f64,f64,f64,f64,f64,f64,f64
"""count""","""32951""","""32341""",32341.0,32341.0,32341.0,32949.0,32949.0,32949.0,32949.0
"""null_count""","""0""","""610""",610.0,610.0,610.0,2.0,2.0,2.0,2.0
"""mean""",null,null,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
"""std""",null,null,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
"""min""","""00066f42aeeb9f3007548bb9d3f33c…","""agro_industria_e_comercio""",5.0,4.0,1.0,0.0,7.0,2.0,6.0
"""25%""",null,null,42.0,339.0,1.0,300.0,18.0,8.0,15.0
"""50%""",null,null,51.0,595.0,1.0,700.0,25.0,13.0,20.0
"""75%""",null,null,57.0,972.0,3.0,1900.0,38.0,21.0,30.0
"""max""","""fffe9eeff12fcbd74a2f2b007dde0c…","""utilidades_domesticas""",76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


In [89]:
check_primary_key(products_df, "product_id")

True

# SELLERS DATASET

- sellers_df:
    - <u>seller_id</u>: char(32)
    - seller_zip_code_prefix: int
    - seller_city: varchar(200) -> fix wrong value by geolocation zip code unique
    - seller_state: char(2)

In [90]:
sellers_df.describe()

statistic,seller_id,seller_zip_code_prefix,seller_city,seller_state
str,str,f64,str,str
"""count""","""3095""",3095.0,"""3095""","""3095"""
"""null_count""","""0""",0.0,"""0""","""0"""
"""mean""",null,32291.059451,null,null
"""std""",null,32713.45383,null,null
"""min""","""0015a82c2db000af6aaaf3ae2ecb05…",1001.0,"""04482255""","""AC"""
"""25%""",null,7094.0,null,null
"""50%""",null,14940.0,null,null
"""75%""",null,65072.0,null,null
"""max""","""ffff564a4f9085cd26170f47323937…",99730.0,"""xaxim""","""SP"""


In [91]:
check_primary_key(sellers_df, "seller_id")

True

In [96]:
print(
    sellers_df.filter(pl.col("seller_city").str.to_integer(strict=False).is_not_null())["seller_id"][0]
)
sellers_df.filter(pl.col("seller_city").str.to_integer(strict=False).is_not_null())

ceb7b4fb9401cd378de7886317ad1b47


seller_id,seller_zip_code_prefix,seller_city,seller_state
str,i64,str,str
"""ceb7b4fb9401cd378de7886317ad1b…",22790,"""04482255""","""RJ"""


In [95]:
geolocation_df.filter(pl.col("geolocation_zip_code_prefix")==22790).unique("geolocation_city")

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
i64,f64,f64,str,str
22790,-23.011335,-43.450256,"""rio de janeiro""","""RJ"""


In [99]:
sellers_df = sellers_df.with_columns(
    seller_city = pl.when(pl.col("seller_city")=="04482255").then(
        pl.lit("rio de janeiro")
    ).otherwise(pl.col("seller_city"))
)
sellers_df.filter(pl.col("seller_id").str.contains("ceb7b4fb9401cd378de7886317ad1b47"))

seller_id,seller_zip_code_prefix,seller_city,seller_state
str,i64,str,str
"""ceb7b4fb9401cd378de7886317ad1b…",22790,"""rio de janeiro""","""RJ"""


# CATEGORY NAME TRANSLATION DATASET:

- category_translation:
    - <u>product_category_name</u>: varchar(200) -> FK product_df.product_category_name
    - product_category_name_english: varchar(200)

In [100]:
category_translation_df.describe()

statistic,product_category_name,product_category_name_english
str,str,str
"""count""","""71""","""71"""
"""null_count""","""0""","""0"""
"""mean""",null,null
"""std""",null,null
"""min""","""agro_industria_e_comercio""","""agro_industry_and_commerce"""
"""25%""",null,null
"""50%""",null,null
"""75%""",null,null
"""max""","""utilidades_domesticas""","""watches_gifts"""


In [101]:
check_primary_key(category_translation_df, "product_category_name")

True